In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-7: Whose samples? Off-policy vs. on-policy

## RL for LLMs, Part 2 (RFT alternative methods), session 7 of 14

## What this session teaches

Every update to a language model learns from some text. The first question is whose text: the model's own samples, or text written by someone else.

PPO and GRPO learn from the model's own samples. SFT learns from a dataset's. In between, a sample from an older copy of the model is almost the model's own.

A second choice is easy to miss. To learn from a token, the update reads the **prefix**, which is the tokens in front of it. It can read the model's own earlier tokens, or tokens imposed from outside.

Each choice has benefits and costs. Imposed text gives a fixed target, but the model never practices from its own mistakes. The model's own text matches what it will face, but it can learn only from what it already produces.

The question to carry: what does each choice buy, and what does it cost?

By the end you can

- say what off-policy and on-policy mean, and place SFT, distillation, PPO and GRPO
- say how stale a batch may be and still count as the model's own
- say what each choice of prefix has as benefits and costs

**Where this session fits: the topics of the series, in order**

<img src="images/rllm7v2/series_strip_6.png" width=88%>

- this session starts topic 6: whose text, teacher-led or student-led (Part 2 begins)


## Setting and words

- We are **fine-tuning** a language model: training it further so its behavior changes
- An **update** is one change to the model's **parameters** (its learned numbers, $\theta$)
- A **sample** is one completion for a prompt, written by the model or by someone else
- Each token in a sample gets a **weight** $w$: the update adds $w$ times the gradient of that token's log probability
  - a weight is not a parameter; this deck never calls parameters "weights"
- A **push** is the resulting change in that token's probability
  - up for a positive weight, down for a negative one, larger for a larger weight
- The **prefix** is the tokens in front of the position being predicted: the prompt plus the completion so far

## The map: two choices, four cells

**Why this slide?** The session has two choices to compare. This picture shows how they fit together, section by section.

> **Two separate choices, whose samples (rows) and which prefix (columns), make a grid of four cells. We fill it in section by section.**

<img src="images/rllm7v2/deck_map_all.png" width=56% alt="Deck map: Dial, Prefix and Benefits-and-costs sections beside a two-by-two grid of samples by prefix.">

- **Dial:** whose samples the update uses, and how stale they may be
- **Prefix:** whether the update reads the model's own earlier tokens, or imposed ones
- **Benefits and costs:** what the cells these two choices make buy and cost


## The three axes in full (previewed in RLLM-6)

<table>
<tr><th>Axis</th><th>The question</th><th>The answers</th></tr>
<tr style="background:#fde3c4"><td><b>1. Whose samples</b></td><td><b>whose completions does the update use?</b></td><td><b>the model's own, or another model's or a dataset's</b></td></tr>
<tr><td>2. Where the weights come from (RLLM-8)</td><td>something to imitate, or to maximize?</td><td>a reference to imitate, or a reward to maximize</td></tr>
<tr><td>3. How fine (RLLM-8)</td><td>how many weights does a sample carry?</td><td>one per completion, one per sampled token, or a distribution per token</td></tr>
</table>

<span style="color:#e8820c">&#9632;</span> <i>orange row, the only bold: the axis this session teaches</i>

- A **reference** is what the update asks the model to reproduce: dataset tokens or a **teacher**, a model whose outputs we imitate
- Not RLLM-5's frozen *reference model*, the KL anchor
- The **prefix** is a separate choice, **not a fourth axis**: the three axes alone place a method
- Next: the **Dial** section, axis 1

## Dial: off-policy and on-policy, not a switch

> **On-policy: the update uses completions the current model just wrote. Off-policy: completions from anywhere else. Between: a dial of staleness.**

**You are here:** Dial (section 1 of 3)

<img src="images/rllm7v2/deck_map_dial.png" width=24% alt="Deck map with the Dial section and the rows of the grid highlighted.">

<img src="images/rllm7v2/dial.png" width=46% alt="A dial from on-policy to off-policy placing REINFORCE, PPO and GRPO, Q-learning and SFT, with the grid rows own and others as brackets.">

- **Recall from A4:** on-policy means the policy that generated the data is the one learned about
- Here a **sample** is one **completion** (the whole output), not an Intro transition. An **update** is one gradient step
- **Stale:** written by an older version of the same model. It describes **the data**, not the loss
- **Dial to grid:** an older copy of the model still counts as **own**. A dataset is **others'**, not a very stale copy

## Dial: how stale is too stale? The ratio and the clip fraction

> **At step 8 the ratio is 1.32, past the clip edge at 1.2. The clip stopped this token's own push, but it is not a bound.**

<img src="images/rllm7v2/ratio_steps.png" width=44% alt="Ratio of current to old probability over update steps passing the clip edge 1.2 at step 8 because other tokens move it.">

- **Recall from B4 and RLLM-5:** the **ratio** is the token's current probability over the old model's
- The clip range is $1\pm\epsilon$ with $\epsilon=0.2$
- Synthetic numbers: ratio = current probability / $0.25$ (the old model's probability for this token)
- With a **positive advantage** the push stops above $1.2$, with a **negative** one below $0.8$
- **Why it passes 1.2:** other tokens' updates share the parameters and keep moving it (RLLM-5: a brake, not a bound)
- **Clip fraction** (named in the companion session P1_ReadingARun): share of tokens whose ratio left the clip range
- **Our judgment:** on-policy enough while most ratios sit near $1$

## Prefix: which prefix does the update condition on?

> **The prefix is the prompt plus the output tokens so far, the state of generation. Teacher forcing: the prefix is the reference's tokens, not the model's own.**
<!-- seed-plant: teacher-forcing-prefix -->

**You are here:** Prefix (section 2 of 3)

<img src="images/rllm7v2/deck_map_prefix.png" width=24% alt="Deck map with the Prefix section and the columns of the grid highlighted.">

<img src="images/rllm7v2/prefix_choice.png" width=46% alt="Two rows of tokens contrasting teacher forcing (reference prefix) with free generation (own prefix).">

- **Recall from C1:** the state is the prompt and the tokens generated so far. That is the prefix
- **Reference**: the tokens the update asks the model to reproduce, from a dataset or a teacher model
- **Free generation:** the prefix is the model's own previous tokens

## Prefix: the two choices crossed

> **Whose samples (rows) and which prefix (columns) are two choices. Crossing them gives four cells, and only the diagonal two have benefits and costs to compare.**

<img src="images/rllm7v2/grid_filled.png" width=50% alt="Two-by-two grid of whose samples by which prefix, with a method or note in each cell.">

- **Recall:** axis 1 is whose samples. Axes 2 and 3 come in RLLM-8
- **What the prefix adds:** samples say whose completions. The prefix says whether the update **imitates** them (teacher forcing) or learns as the model **generates**
- **Others' samples, own prefix: empty.** A completion was written after its author's prefix, not ours
- **Own samples, imposed prefix: degenerate.** Imitating its own samples pushes nowhere on average
- **Next: the benefits and costs of the two diagonal cells**

> **Keep for RLLM-8: own completions with no weights from outside (no reference, no reward) push nowhere on average.**

## Benefits and costs: others' samples with an imposed prefix

**You are here:** Benefits and costs (section 3 of 3)

<img src="images/rllm7v2/deck_map_buys.png" width=14% alt="Deck map with the Benefits-and-costs section and the two diagonal grid cells highlighted.">

<img src="images/rllm7v2/mini_a.png" width=16% alt="Small two-by-two grid with the others-samples imposed-prefix cell highlighted.">

<img src="images/rllm7v2/state_coverage.png" width=20% alt="Region of prefixes the dataset covers, with the model's own path leaving it at a mistake.">

- A **state** is a prefix. The **support** is the region the training data covers (dashed)
- **Benefit:** no generation in the loop, and a fixed, repeatable target
  - a fixed dataset reused for many epochs
  - it can teach what the model would never produce
- **Costs: exposure bias** (debated: how much it matters is not settled): it trains on the dataset's prefixes but generates from its own
  - its own mistake lands on a prefix it was rarely shown
- **Capped by the data:** it cannot beat what the dataset shows
- No signal about which of its own samples are bad

## Benefits and costs: own samples with the model's own prefix

<img src="images/rllm7v2/mini_b.png" width=20% alt="Small two-by-two grid with the own-samples own-prefix cell highlighted.">

<img src="images/rllm7v2/reach_limit.png" width=32% alt="Bar chart of completion probabilities: likely ones are sampled, rare ones give no signal.">

- **Benefit:** trains on the prefixes you will actually visit, so train and test conditioning match
  - it sees its own mistakes, and an external signal can flag them
- **Costs: a reach limit**, you can only directly learn from what you sample (the bars: illustrative)
- Generation sits inside the loop, so sampling is slow

> **Own samples say nothing about what is good: a rater, a verifier or a teacher (a model we imitate) must supply that.**

## Where this returns

> **Whose samples, and which prefix, is the first question every later method answers.**

| Session | Where it meets this |
|---|---|
| **RLLM-8** | own samples with no weights from outside push nowhere on average; where the weights come from |
| **RLLM-9** sample, filter, imitate | the degenerate cell, given a reward that picks which own completions to keep; many steps on one batch: the staleness dial |
| **RLLM-10** DPO | pairs written earlier: off-policy, nearer on-policy if resampled |
| **RLLM-11** on-policy distillation | the own-own cell: the student's samples and prefixes, with the teacher supplying the weights, not the samples |
| **RLLM-13** the map | every method placed by whose samples it uses |

## Think about

<img src="images/rllm7v2/mini_all.png" width=30% alt="Small two-by-two grid of whose samples by which prefix, all four cells shown.">

1. A model fine-tuned to near-zero loss on a clean dataset still writes bad completions when it generates freely. Why?
2. A method trains on another model's saved completions, keeping only those a checker accepts. Which grid cell? What can it **not** teach?
3. Name something an on-policy method cannot teach, however long you train. What would you add?
4. A loop takes eight gradient steps on one fresh batch. How on-policy is step eight, and what would you check (hint: the clip fraction)?

## What we established, and what it sets up

<img src="images/rllm7v2/mini_diag.png" width=30% alt="Small two-by-two grid with both diagonal cells highlighted.">

- **Established**
  - **off-policy** means samples from elsewhere, **on-policy** the model's own, with a staleness dial between (an older copy is own, a dataset is others')
  - the **prefix** is a second choice: imitate the reference's prefix (teacher forcing) or learn on the model's own
  - two grid cells are empty or degenerate, two have benefits and costs to compare
  - others' samples: benefit a fixed target, cost exposure bias
  - own samples: benefit matching conditioning, cost a reach limit
- **Limit:** this says **where samples come from**, not **what the update does with them**
- **Sets up RLLM-8: where do the weights come from?** A reference to imitate or a reward to maximize (axis 2), and how fine (axis 3)

## Reading for this session

- **Suggested** (abstract and introduction for a paper):
  - Kevin Lu et al. (Thinking Machines Lab), *On-Policy Distillation*, blog post, Oct 2025: the opening section
  - Lilian Weng, *Policy Gradient Algorithms*, Lil'Log, 2018: the off-policy section
- More and other cited sources: `reading_list.ipynb`

## Aside: two words Part 2 avoids

- Part 2 never says "score" (the Intro's score function) or "target" (its TD target) for what an update consumes
- We say **reference** and **reward** instead

## Aside: is exposure bias where hallucination comes from?

- **Partly.** Exposure bias is one plausible explanation of why an error persists and compounds ("snowballing")
- It is itself contested
- It does not by itself explain where the first error starts
- How much each cause contributes is an open empirical question
- Zhang et al. (2023), *How Language Model Hallucinations Can Snowball*
- Gekhman et al. (2024), *Does Fine-Tuning LLMs on New Knowledge Encourage Hallucinations?*: evidence for one mechanism each